# 衍生品交易

## 衍生品类型
OKX 提供三种衍生品交易：
- 期货（Futures）
- 永续掉期（Perpetual swaps）
- 期权（Options）

关于解释和更多信息，您可以参考 [衍生品详解](https://www.okx.com/academy/en/bitcoin-derivatives-explained-futures-perpetual-swaps-and-options)。

本教程中我们将以永续掉期为例。

## 导入 API 模块
以下模块可用：
- Trade（交易）
- BlockTrading（大宗交易）
- Funding（资金）
- Account（账户）
- Convert（转换）
- Earning（理财）
- SubAccount（子账户）
- MarketData（行情数据）
- PublicData（公共数据）
- TradingData（交易数据）
- Status（状态）
- NDBroker（NDBroker）
- FDBroker（FDBroker）

In [ ]:
import okx.Trade as Trade

## 填写您的 API 密钥信息

In [ ]:
api_key = "xxxxx"
secret_key = "xxxxx"
passphrase = "xxxxxx"

## 获取可用资金

In [ ]:
import okx.Funding as Funding

flag = "1"  # 实盘：0, 模拟盘：1

fundingAPI = Funding.FundingAPI(api_key, secret_key, passphrase, False, flag)

result = fundingAPI.get_currencies()
print(result)

## 获取市场行情
您也可以将 instType 替换为 "FUTURES" 或 "OPTION" 以获取相应信息。

In [ ]:
import okx.MarketData as MarketData

flag = "1"  # 实盘：0, 模拟盘：1

marketDataAPI = MarketData.MarketAPI(flag = flag)

result = marketDataAPI.get_tickers(instType = "SWAP")
print(result)

# 交易准备
- 确保您了解基本交易规则。请参考 [基本交易规则](https://www.okx.com/support/hc/en-us/sections/360011507312)
- 确保您的交易账户有足够资金。请参考 [获取余额](https://www.okx.com/docs-v5/en/#rest-api-account-get-balance)。

## 获取账户余额。请参考 [获取余额](https://www.okx.com/docs-v5/en/#rest-api-account-get-balance)。

In [ ]:
import okx.Account as Account
flag = "1"  # 实盘：0, 模拟盘：1

accountAPI = Account.AccountAPI(api_key, secret_key, passphrase, False, flag)

result = accountAPI.get_account_balance()
print(result)

## 从 [获取交易产品列表](https://www.okx.com/docs-v5/en/#rest-api-public-data-get-instruments) 获取可用的交易对。
同样地，选择您想要获取信息的 instType。

In [ ]:
import okx.PublicData as PublicData

if __name__ == '__main__':

    flag = "1"  # 实盘：0, 模拟盘：1

    publicDataAPI = PublicData.PublicAPI(flag = flag)

    result = publicDataAPI.get_instruments(instType = "SWAP")
    print(result)

## 从 [获取账户配置](https://www.okx.com/docs-v5/en/#rest-api-account-get-account-configuration) 中的 `acctLv` 参数获取当前账户配置。
在统一交易账户中，正如上个教程提到的，有四种账户模式：
- 简单账户
- 单币种保证金账户
- 多币种保证金账户
- 投资组合保证金账户

只有后三种保证金模式，即单币种保证金、多币种保证金和投资组合保证金才有资格交易衍生品。

In [ ]:
import okx.Account as Account

flag = "1"  # 实盘：0, 模拟盘：1

accountAPI = Account.AccountAPI(api_key, secret_key, passphrase, False, flag)
result = accountAPI.get_account_config()
print(result)

if result['code'] == "0":
    acctLv = result["data"][0]["acctLv"]
    if acctLv == "1":
        print("简单模式")
    elif acctLv == "2":
        print("单币种保证金模式")
    elif acctLv == "3":
        print("多币种保证金模式")
    elif acctLv == "4":
        print("投资组合保证金模式")

## 通过 [设置杠杆](https://www.okx.com/docs-v5/en/#rest-api-account-set-leverage) 设置杠杆。
您可以参考 [OKX 杠杆交易规则](https://www.okx.com/support/hc/en-us/articles/360019908352--OKX-Margin-Trading-Rules)。
单币种杠杆 = 仓位价值 / (全仓仓位中的余额 + 全仓保证金仓位中的未实现盈亏)。
查看 [设置杠杆参考](https://www.okx.com/trade-market/position/swap) 获取更多信息。

In [ ]:
# 为所有全仓 BTC-USDT SWAP 仓位设置杠杆为 5 倍，
# 通过提供任意以 BTC-USDT 为标的的 SWAP instId
result = accountAPI.set_leverage(
    instId = "BTC-USDT-SWAP",
    lever = "5",
    mgnMode = "cross"
)
print(result)

# 为所有逐仓 BTC-USDT SWAP 仓位设置杠杆为 5 倍，
# 通过提供任意以 BTC-USDT 为标的的 SWAP instId
result = accountAPI.set_leverage(
    instId = "BTC-USDT-SWAP",
    lever = "5",
    mgnMode = "isolated"
)
print(result)

# 为逐仓 BTC-USDT-SWAP 多头仓位设置杠杆为 5 倍；
# 这不会影响其他不同到期日或持仓方向（posSide）的 BTC-USDT SWAP 仓位的杠杆
result = accountAPI.set_leverage(
    instId = "BTC-USDT-SWAP",
    lever = "5",
    posSide = "long",
    mgnMode = "isolated"
)
print(result)

# 开始衍生品交易

### 衍生品交易 - 单币种/多币种/投资组合保证金模式

In [ ]:
import okx.Trade as Trade

flag = "1"  # 实盘：0, 模拟盘：1

tradeAPI = Trade.TradeAPI(api_key, secret_key, passphrase, False, flag)

### 在不同持仓（下单）模式下下单：买入/卖出 和 做多/做空
交易期货和永续掉期时有两种持仓（下单）模式：做多/做空 和 买入/卖出（净额）。
您可以通过 API [设置持仓方式](https://www.okx.com/docs-v5/en/#rest-api-account-set-position-mode) 在 做多/做空 和 买入/卖出（净额）之间切换持仓（下单）模式：

In [ ]:
result = accountAPI.set_position_mode(
    posMode="long_short_mode"
)
print(result)

### 通过 [下单](https://www.okx.com/docs-v5/en/#rest-api-trade-place-order) 下限价单

In [ ]:
# 限价单
result = tradeAPI.place_order(
    instId="BTC-USDT-SWAP",
    tdMode="isolated",
    side="buy",
    posSide="net",
    ordType="limit",
    px="19000",
    sz="100"
)
print(result)

if result["code"] == "0":
    print("下单成功，订单 ID = ",result["data"][0]["ordId"])
else:
    print("下单失败，错误码 = ",result["data"][0]["sCode"], ", 错误信息 = ", result["data"][0]["sMsg"])

### 下市价单

In [ ]:
# 市价单
result = tradeAPI.place_order(
    instId="BTC-USDT-SWAP",
    tdMode="isolated",
    side="buy",
    posSide="net",
    ordType="market",
    sz="100"
)
print(result)

if result["code"] == "0":
    print("下单成功，订单 ID = ",result["data"][0]["ordId"])
else:
    print("下单失败，错误码 = ",result["data"][0]["sCode"], ", 错误信息 = ", result["data"][0]["sMsg"])

## 通过 [撤单](https://www.okx.com/docs-v5/en/#rest-api-trade-cancel-order) 撤销订单
您也可以用 clOrdId 代替 ordId

In [ ]:
result = tradeAPI.cancel_order(instId="BTC-USDT-SWAP", ordId="505073046126960640")
print(result)

## 通过 [修改订单](https://www.okx.com/docs-v5/en/#rest-api-trade-amend-order) 修改订单
您也可以用 clOrdId 代替 ordId。
此示例展示修改新的数量。

In [ ]:
result = tradeAPI.amend_order(
    instId="BTC-USDT-SWAP", 
    ordId="505073046126960640",
    newSz="80"
)
print(result)

## 获取订单详情/状态，请参考 [获取订单详情](https://www.okx.com/docs-v5/en/#rest-api-trade-get-order-details)
除了 ordId，您也可以指定 clOrdId 来获取订单详情。

In [ ]:
result = tradeAPI.get_order(instId="BTC-USDT-SWAP", ordId="505073046126960640")
print(result)

## 获取未成交订单列表，请参考 [获取未成交订单列表](https://www.okx.com/docs-v5/en/#rest-api-trade-get-order-list)

In [ ]:
result = tradeAPI.get_order_list()
print(result)

## 获取历史订单，请参考 [获取历史订单（最近 7 天）](https://www.okx.com/docs-v5/en/#rest-api-trade-get-order-history-last-7-days) 和 [获取历史订单（最近 3 个月）](https://www.okx.com/docs-v5/en/#rest-api-trade-get-order-history-last-3-months)

In [ ]:
result = tradeAPI.get_orders_history(
    instType="SPOT"
)
print(result)

In [ ]:
result = tradeAPI.get_orders_history_archive(
    instType="SPOT"
)
print(result)

## 获取历史成交，请参考 [获取历史成交（最近 3 天）](https://www.okx.com/docs-v5/en/#rest-api-trade-get-transaction-details-last-3-days) 和 [获取历史成交（最近 3 个月）](https://www.okx.com/docs-v5/en/#rest-api-trade-get-transaction-details-last-3-months)

In [ ]:
result = tradeAPI.get_fills(
    instType = "SWAP"
)
print(result)

In [ ]:
result = tradeAPI.get_fills_history(
    instType = "SWAP"
)
print(result)

## 通过 [获取持仓信息](https://www.okx.com/docs-v5/en/#rest-api-account-get-positions) 获取持仓信息
当您的账户处于净额模式时，将显示净持仓；当您的账户处于做多/做空模式时，将显示多头或空头持仓。

In [ ]:
result = accountAPI.get_positions()
print(result)

例如，您可以通过响应参数 upl 追踪您的未实现盈亏。